# 3. Avaliação, erros e integração real

Os três sistemas receberam o mesmo teste. Preservamos o corretor automático e uma revisão semântica assistida separada, que expôs falsos positivos e negativos. A revisão não é avaliação humana independente. As células de integração abaixo chamam a API local de verdade; inicie os serviços antes de executá-las.

In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT=Path.cwd()
if not (ROOT/'energia_clara').exists(): ROOT=ROOT.parent
assert (ROOT/'energia_clara').exists()
sys.path.insert(0,str(ROOT))
from energia_clara.config import read_json,read_jsonl
import pandas as pd
from IPython.display import display,Markdown
print('Python:',sys.version.split()[0])


Python: 3.14.7


In [2]:
score=read_json(ROOT/'avaliacao/placar.json')
review=read_json(ROOT/'avaliacao/placar_revisado.json')
for row in review:
    path=ROOT/'avaliacao'/('resultados_'+row['configuracao']+'.jsonl')
    assert hashlib.sha256(path.read_bytes()).hexdigest()==row['saida_sha256']
display(Markdown('## Placar revisado'))
display(pd.DataFrame(review).drop(columns=['saida_sha256']))
display(Markdown('## Placar automático original — não confundir com acurácia semântica'))
display(pd.DataFrame(score))
assert read_json(ROOT/'avaliacao/conclusao_execucao.json')['contextos_b_c_identicos']
print('Contextos de B e C idênticos: verificado na execução.')

## Placar revisado

,configuracao,acertos_fatuais,respondiveis,recusas_corretas,nao_consta,acerto_total_revisado_pct,acerto_factual_revisado_pct,acerto_completo_revisado_pct
0,sem_contexto,0,40,0,20,0.00,0.0,0.00
1,rag,20,40,8,20,46.67,50.0,13.33
2,lora_rag,1,40,11,20,20.00,2.5,18.33


## Placar automático original — não confundir com acurácia semântica

,configuracao,perguntas,chamadas_modelo,acerto_conteudo_pct,acerto_completo_pct,acerto_respondiveis_pct,alucinacao_operacional_pct,recusa_correta_pct,recusa_indevida_pct,json_modelo_valido_pct,json_api_valido_pct,citacao_valida_respondiveis_pct,latencia_media_s,latencia_p50_s,latencia_p95_s,recall_1,recall_1_sem_limiar,recall_3,recall_3_sem_limiar
0,sem_contexto,60,60,3.33,0.00,5.0,90.00,0.0,10.0,0.00,100.0,NaN,3.675,4.363,4.547,NaN,NaN,NaN,NaN
1,rag,60,52,16.67,13.33,5.0,83.33,40.0,0.0,0.00,100.0,0.0,3.690,4.574,5.125,0.9,0.9,1.0,1.0
2,lora_rag,60,52,28.33,25.00,17.5,65.00,50.0,10.0,94.23,100.0,85.0,1.807,1.729,3.194,0.9,0.9,1.0,1.0


Contextos de B e C idênticos: verificado na execução.


In [3]:
errors=read_json(ROOT/'avaliacao/cinco_erros.json')
for i,e in enumerate(errors,1):
    display(Markdown(f"### Erro {i}: {e['categoria']} — {e['id']}"))
    print('Pergunta:',e['pergunta'])
    print('Esperado:',e['resposta_esperada'])
    print('Saída real:',e['saida_bruta'] or e['resposta_pipeline'])
    print('Causa provável:',e['causa_provavel'])
    print('Correção proposta:',e['correcao_proposta'])

### Erro 1: Conteúdo divergente do gabarito — T-E015-1

Pergunta: Onde posso consultar o consumo de um refrigerador registrado no Inmetro?
Esperado: Consulte o Sistema PBE do Inmetro e filtre por marca, modelo e capacidade.
Saída real: {"status":"respondido","resposta":"Não. Consultar consumo em kWh por mês é uma vez.","fontes":["E015"]}
Causa provável: O modelo não preservou os fatos exigidos ou acrescentou número sem sustentação.
Correção proposta: Inspecionar cobertura e posição do trecho correto e ampliar exemplos de extração fundamentada; revisar também possíveis falsos negativos do corretor.


### Erro 2: Saída fora do contrato JSON — T-NC-09

Pergunta: Qual é a potência do meu chuveiro sem informar o modelo?
Esperado: Não consta na base de conhecimento.
Saída real: Qual é a potência do meu chuveiro sem informar o modelo?
Causa provável: A geração livre não manteve o esquema esperado; a API devolveu erro verificável.
Correção proposta: Testar decodificação restrita por esquema e medir sua contribuição separadamente do LoRA; preservar a taxa bruta de falhas de formato.


### Erro 3: Saída fora do contrato JSON — T-E015-1

Pergunta: Onde posso consultar o consumo de um refrigerador registrado no Inmetro?
Esperado: Consulte o Sistema PBE do Inmetro e filtre por marca, modelo e capacidade.
Saída real: Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrad
Causa provável: A geração livre não manteve o esquema esperado; a API devolveu erro verificável.
Correção proposta: Testar decodificação restrita por esquema e medir sua contribuição separadamente do LoRA; preservar a taxa bruta de falhas de formato.


### Erro 4: Saída fora do contrato JSON — T-NC-09

Pergunta: Qual é a potência do meu chuveiro sem informar o modelo?
Esperado: Não consta na base de conhecimento.
Saída real: {"status":"respondido","resposta":"Potência e energia; potência e energia; potência e energia; potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energia; Potência e energ
Causa provável: A geração livre não manteve o esquema esperado; a API devolveu erro verificável.
Correção proposta: Testar decodificação restrita por esquema e medir sua contribuição separadamente do LoRA; preservar a taxa bruta de falhas de formato.


### Erro 5: Saída fora do contrato JSON — T-E015-1

Pergunta: Onde posso consultar o consumo de um refrigerador registrado no Inmetro?
Esperado: Consulte o Sistema PBE do Inmetro e filtre por marca, modelo e capacidade.
Saída real: Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrado no Inmetro?

Onde posso consultar o consumo de um refrigerador registrad
Causa provável: A geração livre não manteve o esquema esperado; a API devolveu erro verificável.
Correção proposta: Testar decodificação restrita por esquema e medir sua contribuição separadamente do LoRA; preservar a taxa bruta de falhas de formato.


In [4]:
import requests
session=requests.Session(); session.trust_env=False
API='http://127.0.0.1:8000'
health=session.get(API+'/saude',timeout=10); health.raise_for_status()
print('API real:',health.json())
with (ROOT/'dados/consumo_exemplo.csv').open('rb') as f:
    uploaded=session.post(API+'/ingerir-consumo',files={'arquivo':('consumo.csv',f,'text/csv')},timeout=10)
uploaded.raise_for_status()
assert uploaded.json()['registros']==8
sim=session.post(API+'/simular',json={'aparelhos':uploaded.json()['dados'],'tarifa_kwh':0.95},timeout=10)
sim.raise_for_status()
assert sim.json()['consumo_total_kwh']==229.84
print('Simulação via API:',sim.json())
source=session.post(API+'/consultar-base',json={'pergunta':'Como economizar energia com o chuveiro?'},timeout=180)
source.raise_for_status()
print('Consulta documental via API:',source.json())
experiment=session.post(API+'/perguntar',json={'pergunta':'Aderir à Tarifa Branca sempre deixa a conta mais barata?','configuracao':'lora_rag'},timeout=180)
experiment.raise_for_status()
print('Geração LoRA via API:',experiment.json())
evidence={'saude':health.json(),'ingestao_registros':uploaded.json()['registros'],'simulacao':sim.json(),'consulta_base':source.json(),'geracao_experimental':experiment.json(),
          'nota':'A pergunta da Tarifa Branca é da família de treino; demonstra integração, não generalização. Não compõe o placar final.'}
from energia_clara.config import write_json
write_json(ROOT/'avaliacao/demonstracao_api.json',evidence)

API real: {'status': 'ok', 'modelo': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'modelo_local': True, 'encoder_local': True, 'adaptador_treinado': True, 'avaliacao_concluida': True, 'persistencia_perguntas': False, 'servico': 'local'}
Simulação via API: {'aparelhos': [{'aparelho': 'Chuveiro', 'potencia_w': 5500.0, 'horas_dia': 0.5, 'dias_mes': 30.0, 'quantidade': 1, 'fator_uso': 1.0, 'consumo_kwh': 82.5, 'custo_energia_estimado': 78.38}, {'aparelho': 'Ar-condicionado', 'potencia_w': 900.0, 'horas_dia': 6.0, 'dias_mes': 20.0, 'quantidade': 1, 'fator_uso': 0.65, 'consumo_kwh': 70.2, 'custo_energia_estimado': 66.69}, {'aparelho': 'Geladeira', 'potencia_w': 150.0, 'horas_dia': 24.0, 'dias_mes': 30.0, 'quantidade': 1, 'fator_uso': 0.3, 'consumo_kwh': 32.4, 'custo_energia_estimado': 30.78}, {'aparelho': 'Televisor', 'potencia_w': 100.0, 'horas_dia': 4.0, 'dias_mes': 30.0, 'quantidade': 1, 'fator_uso': 1.0, 'consumo_kwh': 12.0, 'custo_energia_estimado': 11.4}, {'aparelho': 'Notebook', 'potencia_w

Consulta documental via API: {'status': 'fontes_encontradas', 'resposta': 'Encontrei estas orientações relacionadas na base. Elas não determinam valores ou condições da sua casa.', 'fontes': ['E019', 'E018', 'E027'], 'evidencias': [{'id': 'E019', 'fonte_id': 'S07', 'titulo': 'Tempo de banho', 'texto': 'Reduza o tempo de banho e feche a água enquanto se ensaboa para diminuir o tempo de funcionamento do chuveiro elétrico.', 'secao': 'Chuveiro', 'url': 'https://www.cemig.com.br/noticia/dicas-e-orientacoes/aprenda-o-basico-da-economia-de-energia-em-casa/', 'entidade': 'Cemig', 'consultado_em': '2026-11-06', 'tipo': 'nota_autoral_fundamentada', 'grupo_fato': 'E019', 'particao_qa': 'teste', 'score': 0.8976702690124512}, {'id': 'E018', 'fonte_id': 'S07', 'titulo': 'Potência do chuveiro', 'texto': 'Quando for confortável e compatível com o manual, prefira a posição verão ou menor aquecimento do chuveiro. A economia depende do aparelho e uso.', 'secao': 'Chuveiro', 'url': 'https://www.cemig.com

Geração LoRA via API: {'status': 'respondido', 'resposta': 'Não. A tarifa branca pode aumentar a conta se o consumo fica concentrado em ponta e intermediário.', 'fontes': ['E010'], 'configuracao': 'lora_rag', 'origem': 'modelo', 'modelo_chamado': True, 'json_modelo_valido': True, 'erro_verificacao': '', 'tempo_s': 2.021, 'evidencias': [{'id': 'E010', 'fonte_id': 'S04', 'titulo': 'Escolha da Tarifa Branca', 'texto': 'A Tarifa Branca pode aumentar a conta quando o consumo fica concentrado em ponta e intermediário. A escolha exige analisar o perfil.', 'secao': 'Quais os benefícios da Tarifa Branca?', 'url': 'https://www.gov.br/aneel/pt-br/assuntos/tarifas/tarifa-branca/perguntas-frequentes-sobre-tarifa-branca/', 'entidade': 'ANEEL', 'consultado_em': '2026-11-06', 'tipo': 'nota_autoral_fundamentada', 'grupo_fato': 'E010', 'particao_qa': 'treino', 'score': 0.9028428792953491}]}


## Interpretação

LoRA melhorou JSON e parte das recusas, mas não sustentou ganho factual frente ao RAG simples. O produto prioriza apresentar as notas com fontes diretamente. Essa consulta documental não foi creditada como geração ou como resultado do treinamento. O objetivo de uso livre em produção requer novo ciclo de dados e validação.